# 03 · Linhas de base: regras e TF-IDF

Treina TF-IDF + Regressão Logística / Random Forest nos rótulos fracos do LLM e avalia no teste do gabarito.

In [ ]:
# Configuração: no Colab, clona o repositório e instala o pacote `absa`
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[nlp]"
    # Guarda cache do LLM e modelos no Drive para sobreviver à queda da sessão
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models

In [ ]:
import pandas as pd
from absa.config import ANNOTATED, ASPECTS, INTERIM, LABEL_COLUMNS
from absa.heuristics import heuristic_labels
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.models.baseline import TfidfAspectClassifier
from absa.preprocess import clean_classic
from absa.sampling import load_gold

dev, test = load_gold()
fracos = pd.read_csv(INTERIM / "rotulos_llm.csv")  # gerado no notebook 04
print(len(fracos), len(dev), len(test))

## Baseline 0: regras

In [ ]:
regras = pd.DataFrame([heuristic_labels(t, a) for t, a in zip(test.texto, test.atrasado)])
regras.columns = LABEL_COLUMNS
print("F1 tupla:", round(tuple_f1(test, regras), 3))
per_aspect_report(test, regras).round(3)

## Baseline 1: TF-IDF + Regressão Logística / Random Forest

In [ ]:
# Opcional: lematização com spaCy (python -m spacy download pt_core_news_sm)
X_train = fracos.texto.map(clean_classic)
X_test = test.texto.map(clean_classic)

resultados = {}
for nome in ["logreg", "rf"]:
    clf = TfidfAspectClassifier(nome).fit(X_train, fracos)
    pred = clf.predict(X_test)
    resultados[nome] = pred
    f1, lo, hi = bootstrap_ci(test, pred)
    print(f"{nome}: F1 macro {f1:.3f} (IC95% {lo:.3f}–{hi:.3f}) | F1 tupla {tuple_f1(test, pred):.3f}")
per_aspect_report(test, resultados["logreg"]).round(3)

In [ ]:
# Termos mais associados a "entrega negativo" (explicabilidade global)
clf = TfidfAspectClassifier("logreg").fit(X_train, fracos)
clf.top_terms("entrega", "negativo")

In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)
for nome, pred in resultados.items():
    pred.assign(review_id=test.review_id.values).to_csv(INTERIM / f"pred_teste_{nome}.csv", index=False)